## **HealthCare Location Intelligence**

**Problem Statement & Approach**

Selecting a location for a new physiotherapy clinic in the GTA requires more than high-level population or income statistics. Most neighbourhoods already have existing clinics, making it necessary to evaluate local demographics, competition, and referral support at a granular geographic level. These data points exist across multiple sources but are not directly linked.

In this notebook, we build a geospatial data pipeline to address this problem by loading census shape and boundary files, fetching demographic data, and retrieving physiotherapy clinics and nearby support facilities via API calls. We then use spatial joins to align all entities to common geographic units, creating a unified dataset that enables location-level analysis and informed decision-making.

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#import necessary libraries
import os
import time
import requests
import pandas as pd

 # **Data Collection**

We will use google places API to query physiotherapy clinics across the 4 GTA locations the client is interested in: Mississauga, Burlington, Oakville and Etobicoke

In [ ]:
# Declaring API key for google places
API_KEY = "YOUR API KEY"
# Path to save files
save_path = "/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy"

In [ ]:
# Queries for physiotherapy
queries = {
    "physio": [
        "physiotherapy clinic",
        "physiotherapist",
        "rehabilitation clinic",
        "sports medicine clinic",
        "wellness clinic"
    ]
}


In [ ]:
# Co-ordinates for GTA cities requested by client
city_coords = {
    "Mississauga": [
        "43.5890,-79.6441", "43.6000,-79.7000", "43.5500,-79.6500",
        "43.5700,-79.6000", "43.6200,-79.5800", "43.5800,-79.7300",
        "43.5400,-79.6700", "43.6100,-79.6200"
    ],
    "Oakville": [
        "43.4675,-79.6877", "43.4500,-79.7100", "43.4800,-79.6400",
        "43.4300,-79.6500", "43.5000,-79.6700", "43.4600,-79.6200",
        "43.4900,-79.7000", "43.4450,-79.6850"
    ],
    "Burlington": [
        "43.3255,-79.7990", "43.3400,-79.8000", "43.3500,-79.7700",
        "43.3100,-79.7500", "43.3750,-79.8100", "43.3650,-79.7900",
        "43.3300,-79.7400", "43.3050,-79.7200"
    ],
    "Etobicoke": [
        "43.6205,-79.5132", "43.6000,-79.5500", "43.6300,-79.5200",
        "43.6400,-79.5000", "43.6100,-79.4800", "43.5850,-79.5200",
        "43.5900,-79.5000", "43.6250,-79.5400"
    ]
}

To capture the full physiotherapy ecosystem, we will use multiple search queries, including:

- Physiotherapy clinics
- Rehabilitation clinics
- Sports medicine and wellness centers

For each query and location:

- Google Places Text Search API will be used to retrieve candidate locations
- Pagination will be handled to ensure no results are missed
- Place Details API calls will be made to extract accurate coordinates, ratings, and metadata

In [ ]:
# URL for text and detail search
TEXT_URL = "https://maps.googleapis.com/maps/api/place/textsearch/json"
DETAILS_URL = "https://maps.googleapis.com/maps/api/place/details/json"


In [ ]:
# Function to query places
def get_places(query, location, radius=5000):
    params = {
        "query": query,
        "location": location,
        "radius": radius,
        "key": API_KEY
    }
    results = []
    while True:
        resp = requests.get(TEXT_URL, params=params)
        data = resp.json()
        if data.get("status") not in ("OK","ZERO_RESULTS"):
            break
        results.extend(data.get("results", []))
        if "next_page_token" not in data:
            break
        time.sleep(2)  # wait before using next_page_token
        params = {"pagetoken": data["next_page_token"], "key": API_KEY}
    return results

# Function to get details for each place id
def get_place_details(place_id):
    params = {
        "place_id": place_id,
        "fields": "place_id,name,formatted_address,geometry,types,"
                  "rating,user_ratings_total,website,url,reviews",
        "key": API_KEY
    }
    resp = requests.get(DETAILS_URL, params=params)
    return resp.json().get("result", {})

# Function to filter if clinic offers physio services
def is_physio(details):
    types = details.get("types", [])
    name = details.get("name","").lower()
    addr = details.get("formatted_address","").lower()
    physio_terms = ["physio","physiotherapy","physiotherapist"]

    # Broad check — keep if Google tags it broadly as medical
    if any(t in types for t in physio_terms):
        return True

    # Check name for physio-like words
    if any(term in name for term in physio_terms):
        return True

    # Check address for physio-like words
    if any(term in addr for term in physio_terms):
        return True

    # Check reviews text
    for r in details.get("reviews", []):
        if any(term in r.get("text","").lower() for term in physio_terms):
            return True

    return False


In [ ]:
# Main loop to pull relevant physiotherapy clinics
all_clinics = []
all_reviews = []

for city, coords_list in city_coords.items():
    print(f"\n=== Collecting for {city} ===")
    seen_ids = set()

    for q in queries["physio"]:
        print(f"  • Query: {q}")
        for loc in coords_list:
            results = get_places(q, loc)
            for r in results:
                pid = r.get("place_id")
                if not pid or pid in seen_ids:
                    continue
                seen_ids.add(pid)

                details = get_place_details(pid)
                if not is_physio(details):
                    continue  # skip non-physio

                # Save clinic
                locn = details.get("geometry",{}).get("location",{})
                all_clinics.append({
                    "City": city,
                    "Place ID": pid,
                    "Name": details.get("name"),
                    "Address": details.get("formatted_address"),
                    "Latitude": locn.get("lat"),
                    "Longitude": locn.get("lng"),
                    "Rating": details.get("rating"),
                    "User Ratings Total": details.get("user_ratings_total"),
                    "Types": ",".join(details.get("types",[])),
                    "Website": details.get("website"),
                    "Google Maps URL": details.get("url")
                })

                # Save reviews
                for rv in details.get("reviews", []):
                    all_reviews.append({
                        "City": city,
                        "Place ID": pid,
                        "Clinic Name": details.get("name"),
                        "Author": rv.get("author_name"),
                        "Rating": rv.get("rating"),
                        "Text": rv.get("text"),
                        "Time": rv.get("time"),
                        "Relative Time": rv.get("relative_time_description")
                    })
            time.sleep(1)

# Function to save output
os.makedirs(save_path, exist_ok=True)
df_clinics = pd.DataFrame(all_clinics).drop_duplicates(subset="Place ID")
df_reviews = pd.DataFrame(all_reviews)

df_clinics.to_csv(os.path.join(save_path, "physio_clinics.csv"), index=False)
df_reviews.to_csv(os.path.join(save_path, "physio_clinic_reviews.csv"), index=False)

print(f"\n Saved {len(df_clinics)} clinics and {len(df_reviews)} reviews")



=== Collecting for Mississauga ===
  • Query: physiotherapy clinic
  • Query: physiotherapist
  • Query: rehabilitation clinic
  • Query: sports medicine clinic
  • Query: wellness clinic

=== Collecting for Oakville ===
  • Query: physiotherapy clinic
  • Query: physiotherapist
  • Query: rehabilitation clinic
  • Query: sports medicine clinic
  • Query: wellness clinic

=== Collecting for Burlington ===
  • Query: physiotherapy clinic
  • Query: physiotherapist
  • Query: rehabilitation clinic
  • Query: sports medicine clinic
  • Query: wellness clinic

=== Collecting for Etobicoke ===
  • Query: physiotherapy clinic
  • Query: physiotherapist
  • Query: rehabilitation clinic
  • Query: sports medicine clinic
  • Query: wellness clinic

✅ Saved 957 clinics and 6839 reviews


**Observations:** A total of 957 Clincs along with 6839 reviews were fetched for all 4 GTA locations combined

In [ ]:
# Reading the clinics data
pc_df=pd.read_csv("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/physio_clinics.csv")
pc_df.head()

,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL
0,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,"1224 Dundas St W, Mississauga, ON L5C 4G7, Canada",43.553812,-79.645781,4.8,73.0,"establishment,health,point_of_interest",https://real-rehab.com/,https://maps.google.com/?cid=4686936157919423019
1,Mississauga,ChIJC7CuKt81K4gRorDvzCPO2bY,Vibrant Physiotherapy,"3050 Confederation Pkwy #206, Mississauga, ON ...",43.577976,-79.621476,4.9,36.0,"establishment,health,point_of_interest",http://www.vibrantphysiotherapy.ca/,https://maps.google.com/?cid=13175788838006534306
2,Mississauga,ChIJgb1mccFHK4gR_6BEh0rlLJ0,Proremedy Physiotherapy,"190 Robert Speck Pkwy Suite 200, Mississauga, ...",43.596916,-79.632251,4.9,171.0,"establishment,health,physiotherapist,point_of_...",https://proremedyphysio.com/physiotherapy-miss...,https://maps.google.com/?cid=11325679271189717247
3,Mississauga,ChIJkZx3yb1DK4gRU2szteXkKgc,Smart Physiotherapy Clinic,"1151 Dundas St W, Mississauga, ON L5C 1C4, Canada",43.557562,-79.645423,2.5,31.0,"establishment,health,point_of_interest",http://www.paradisewelness.ca/,https://maps.google.com/?cid=516476782526032723
4,Mississauga,ChIJZ49FH3Q4K4gRMECgKM4_lYE,Maxwell Physiotherapy and Rehab Clinic,"3415 Fieldgate Dr, Mississauga, ON L4X 2J4, Ca...",43.624210,-79.588187,4.9,143.0,"establishment,health,physiotherapist,point_of_...",http://www.maxwellclinic.ca/,https://maps.google.com/?cid=9337439557099995184


In [ ]:
# checking shape for clinic data
pc_df.shape

(957, 11)

In [ ]:
# Read the reviews data
pcr_df=pd.read_csv("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/physio_clinic_reviews.csv")
pcr_df.head()

,City,Place ID,Clinic Name,Author,Rating,Text,Time,Relative Time
0,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Deepak Kamani,5,I had a wonderful experience at Real rehab. Th...,1735358537,9 months ago
1,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Deep Inder,5,I can’t say enough good things about my massag...,1745077110,5 months ago
2,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Frankline Antony,5,“I had a fantastic experience.The physiotherap...,1751733894,2 months ago
3,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Katie C,5,I recently started seeing Reva for massage the...,1743888183,5 months ago
4,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Monika Meharchandani,5,Visiting Real Rehab is always an exceptional e...,1742052614,6 months ago


In [ ]:
# checking shape for reviews data
pcr_df.shape

(6839, 8)

We will similarly query locations for support facilities such as hospitals, medical centers and walk-in-clinics. These sources are important drivers for patient referrals to Physiotherapy clinics

In [ ]:
# creating queries for support facilities.
queries = {
    "support": [
        "hospital",
        "medical centre",
        "medical center",
        "walk-in clinic",
        "doctor clinic",
        "family physician"
    ]
}


In [ ]:
# function to filter if clinic offers medical services
def is_medical_facility(details):
    types = details.get("types", [])
    name = details.get("name", "").lower()
    addr = details.get("formatted_address", "").lower()
    reviews = details.get("reviews", [])

    medical_terms = [
        "hospital", "medical", "doctor", "physician",
        "clinic", "health centre", "health center"
    ]

    #  Check types
    if any(term in ",".join(types).lower() for term in medical_terms):
        return True

    #  Check name/address
    if any(term in name for term in medical_terms):
        return True
    if any(term in addr for term in medical_terms):
        return True

    #  Check reviews
    for r in reviews:
        if any(term in r.get("text", "").lower() for term in medical_terms):
            return True

    return False


In [ ]:
# Main loop to fetch support facilities
all_support = []
all_support_reviews = []

for city, coords_list in city_coords.items():
    print(f"\n=== Collecting Support Facilities for {city} ===")
    seen_ids = set()

    for q in queries["support"]:
        print(f"  • Query: {q}")
        for loc in coords_list:
            results = get_places(q, loc)
            for r in results:
                pid = r.get("place_id")
                if not pid or pid in seen_ids:
                    continue
                seen_ids.add(pid)

                details = get_place_details(pid)
                if not is_medical_facility(details):
                    continue  # skip non-medical

                # Save facility
                locn = details.get("geometry", {}).get("location", {})
                all_support.append({
                    "City": city,
                    "Place ID": pid,
                    "Name": details.get("name"),
                    "Address": details.get("formatted_address"),
                    "Latitude": locn.get("lat"),
                    "Longitude": locn.get("lng"),
                    "Rating": details.get("rating"),
                    "User Ratings Total": details.get("user_ratings_total"),
                    "Types": ",".join(details.get("types", [])),
                    "Website": details.get("website"),
                    "Google Maps URL": details.get("url")
                })

                # Save reviews
                for rv in details.get("reviews", []):
                    all_support_reviews.append({
                        "City": city,
                        "Place ID": pid,
                        "Facility Name": details.get("name"),
                        "Author": rv.get("author_name"),
                        "Rating": rv.get("rating"),
                        "Text": rv.get("text"),
                        "Time": rv.get("time"),
                        "Relative Time": rv.get("relative_time_description")
                    })
            time.sleep(1)

# Save outputs
df_support = pd.DataFrame(all_support).drop_duplicates(subset="Place ID")
df_support_reviews = pd.DataFrame(all_support_reviews)

df_support.to_csv(os.path.join(save_path, "support_facilities.csv"), index=False)
df_support_reviews.to_csv(os.path.join(save_path, "support_facility_reviews.csv"), index=False)

print(f"\n✅ Saved {len(df_support)} support facilities and {len(df_support_reviews)} reviews")



=== Collecting Support Facilities for Mississauga ===
  • Query: hospital
  • Query: medical centre
  • Query: medical center
  • Query: walk-in clinic
  • Query: doctor clinic
  • Query: family physician

=== Collecting Support Facilities for Oakville ===
  • Query: hospital
  • Query: medical centre
  • Query: medical center
  • Query: walk-in clinic
  • Query: doctor clinic
  • Query: family physician

=== Collecting Support Facilities for Burlington ===
  • Query: hospital
  • Query: medical centre
  • Query: medical center
  • Query: walk-in clinic
  • Query: doctor clinic
  • Query: family physician

=== Collecting Support Facilities for Etobicoke ===
  • Query: hospital
  • Query: medical centre
  • Query: medical center
  • Query: walk-in clinic
  • Query: doctor clinic
  • Query: family physician

✅ Saved 1008 support facilities and 6709 reviews


**Observations:** A total of 1008 support facilities and 6709 reviews were fetched across all 4 GTA locations

In [ ]:
#loading support facilities dataset
sf_df=pd.read_csv("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/support_facilities.csv")
sf_df.head()

,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL
0,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,"2200 Eglinton Ave W, Mississauga, ON L5M 2N1, ...",43.558609,-79.703268,3.0,1649.0,"establishment,health,hospital,point_of_interest",http://trilliumhealthpartners.ca/,https://maps.google.com/?cid=2655445724276753516
1,Mississauga,ChIJDYSaI_ZGK4gRHtlzIG1irZU,Mississauga Hospital,"100 Queensway W, Mississauga, ON L5B 1B8, Canada",43.571699,-79.607584,2.6,1378.0,"establishment,health,hospital,point_of_interest",https://www.thp.ca/,https://maps.google.com/?cid=10785384903457626398
2,Mississauga,ChIJrSXDN6JnK4gR_Z2rizri6Ug,Oakville Trafalgar Memorial Hospital,"3001 Hospital Gate, Oakville, ON L6M 0L8, Canada",43.450353,-79.764738,2.9,1306.0,"establishment,health,hospital,point_of_interest",https://www.haltonhealthcare.on.ca/locations/o...,https://maps.google.com/?cid=5253979181383654909
3,Mississauga,ChIJnTiQbHUxK4gRfP8pK7aXbxM,Humber River Hospital,"1235 Wilson Ave, North York, ON M3M 0B2, Canada",43.724222,-79.488578,2.6,2281.0,"establishment,health,hospital,point_of_interest",http://www.hrh.ca/,https://maps.google.com/?cid=1400504817799528316
4,Mississauga,ChIJa00ZpMk1K4gRVd9G-3qal48,St. Joseph's Health Centre,"30 The Queensway, Toronto, ON M6R 1B5, Canada",43.640165,-79.450064,3.2,1339.0,"establishment,health,hospital,point_of_interest",https://unityhealth.to/,https://maps.google.com/?cid=10346908521899417429


In [ ]:
# checking shape of data for support facilities
sf_df.shape

(1008, 11)

In [ ]:
#laoding dataset for support facilities reviews
sfr_df=pd.read_csv("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/support_facility_reviews.csv")
sfr_df.head()


,City,Place ID,Facility Name,Author,Rating,Text,Time,Relative Time
0,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Tejal P,4,Nurses here are angels. Had a great experience...,1753800941,2 months ago
1,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Ram Venkat,4,My daughters delivery 27th July 2025\nGreat se...,1758075027,a week ago
2,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Ozair Mohammad,1,"Insane wait times, hasn’t gotten better in yea...",1749294804,3 months ago
3,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Sharry A,5,I had an emergency visit recently and was incr...,1746054361,5 months ago
4,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Terrance 413,5,No one likes spending 6.5 hours in the Emergen...,1753756252,2 months ago


We will use Official **Statistics Canada shapefiles**  to establish geographic context:

- Census tract boundaries
- City-level administrative boundaries

All shapefiles will be:

- Reprojected to a common coordinate system (WGS84)
- Cleaned and standardized
- Spatially filtered to only include census tracts intersecting with the target GTA regions

This step will ensure that all downstream joins and aggregations will be geographically consistent.

In [ ]:
# checking shape of data for support facilities reviews
sfr_df.shape

(6709, 8)

In [ ]:
# Defining path for census tracts shape file and city boundary shape file
path_census_tracts="/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/census tracts/Copy of lct_000a21a_e.shp"
path_city_boundaries="/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/City Boundaries/Copy of lcsd000a24a_e.shp"

In [ ]:
# loading census tracts & city boundaries & filtering down to 4 cities on census tracts.
import geopandas as gpd
# Load Census Tract shapefile ---
gdf_tracts = gpd.read_file(path_census_tracts)

# Set CRS if missing and reproject to EPSG:4326
if gdf_tracts.crs is None:
    gdf_tracts.set_crs(epsg=3347, inplace=True)
gdf_tracts = gdf_tracts.to_crs(epsg=4326)

# Load City Boundaries shapefile ---
gdf_cities = gpd.read_file(path_city_boundaries)

# Reproject to same CRS as census tracts
if gdf_cities.crs != gdf_tracts.crs:
    gdf_cities = gdf_cities.to_crs(gdf_tracts.crs)

# Rename CSDNAME to City for clarity
gdf_cities = gdf_cities.rename(columns={"CSDNAME": "City"})

# Filter only 4 target cities ---
target_cities = ['Mississauga', 'Burlington', 'Oakville', 'Toronto']
gdf_cities_filtered = gdf_cities[gdf_cities['City'].isin(target_cities)]

# Step 4: Spatial join to get only tracts intersecting with city polygons ---
gdf_filtered_with_ids = gpd.sjoin(gdf_tracts, gdf_cities_filtered, how="inner", predicate="intersects")

# Final output
print("Filtered CTUIDs:", gdf_filtered_with_ids['CTUID'].nunique())
gdf_filtered_with_ids.head()


Filtered CTUIDs: 853


,CTUID,DGUID,CTNAME,LANDAREA,PRUID_left,geometry,index_right,CSDUID,City,CSDTYPE,PRUID_right,PRNAME,CDUID,CDNAME,CDTYPE
25,5370140.04,2021S05075370140.04,0140.04,3.0270,35,"POLYGON ((-79.89286 43.32909, -79.89226 43.328...",1847,3524002,Burlington,CY,35,Ontario,3524,Halton,RM
113,5350516.31,2021S05075350516.31,0516.31,1.4206,35,"POLYGON ((-79.69225 43.56952, -79.69207 43.569...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
118,5350516.32,2021S05075350516.32,0516.32,1.0444,35,"POLYGON ((-79.78141 43.57025, -79.78486 43.567...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
119,5350520.07,2021S05075350520.07,0520.07,0.3389,35,"POLYGON ((-79.61896 43.57806, -79.61944 43.577...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
120,5350520.08,2021S05075350520.08,0520.08,1.1020,35,"POLYGON ((-79.62281 43.57841, -79.62241 43.578...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM


We will now convert Clinic and hospital coordinates into GeoDataFrames and map to census tracts using **point-in-polygon spatial joins**.

Each clinic and facility will be assigned:

- A **DGUID** (primary census identifier)
- A census tract ID

Any locations that do not fall cleanly within a census boundary will be excluded to maintain spatial accuracy. This mapping step is critical—it enables direct linkage between real-world locations and demographic data.

In [ ]:
# Mapping physiotherapy locations to census tracts
from shapely.geometry import Point
# Convert physio merged DataFrame to GeoDataFrame
geometry = [Point(xy) for xy in zip(pc_df["Longitude"], pc_df["Latitude"])]
gdf_physio = gpd.GeoDataFrame(pc_df.copy(), geometry=geometry, crs="EPSG:4326")

# Making sure the census tract data is in the same CRS
gdf_filtered_with_ids = gdf_filtered_with_ids.to_crs("EPSG:4326")

# Spatial join to get DGUID and other tract details
gdf_physio_DGUID = gpd.sjoin(
    gdf_physio,
    gdf_filtered_with_ids[["DGUID", "CTUID", "geometry"]],
    how="left",
    predicate="within"
)

# Drop unmatched rows
gdf_physio_DGUID = gdf_physio_DGUID.dropna(subset=["DGUID"])

# Checking Head
print(f" Total matched physiotherapy clinics: {gdf_physio_DGUID.shape[0]}")
gdf_physio_DGUID.head()


✅ Total matched physiotherapy clinics: 968


,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL,geometry,index_right,DGUID,CTUID
0,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,"1224 Dundas St W, Mississauga, ON L5C 4G7, Canada",43.553812,-79.645781,4.8,73.0,"establishment,health,point_of_interest",https://real-rehab.com/,https://maps.google.com/?cid=4686936157919423019,POINT (-79.64578 43.55381),3265.0,2021S05075350513.01,5350513.01
1,Mississauga,ChIJC7CuKt81K4gRorDvzCPO2bY,Vibrant Physiotherapy,"3050 Confederation Pkwy #206, Mississauga, ON ...",43.577976,-79.621476,4.9,36.0,"establishment,health,point_of_interest",http://www.vibrantphysiotherapy.ca/,https://maps.google.com/?cid=13175788838006534306,POINT (-79.62148 43.57798),120.0,2021S05075350520.08,5350520.08
2,Mississauga,ChIJgb1mccFHK4gR_6BEh0rlLJ0,Proremedy Physiotherapy,"190 Robert Speck Pkwy Suite 200, Mississauga, ...",43.596916,-79.632251,4.9,171.0,"establishment,health,physiotherapist,point_of_...",https://proremedyphysio.com/physiotherapy-miss...,https://maps.google.com/?cid=11325679271189717247,POINT (-79.63225 43.59692),4827.0,2021S05075350527.13,5350527.13
3,Mississauga,ChIJkZx3yb1DK4gRU2szteXkKgc,Smart Physiotherapy Clinic,"1151 Dundas St W, Mississauga, ON L5C 1C4, Canada",43.557562,-79.645423,2.5,31.0,"establishment,health,point_of_interest",http://www.paradisewelness.ca/,https://maps.google.com/?cid=516476782526032723,POINT (-79.64542 43.55756),1240.0,2021S05075350518.00,5350518.00
4,Mississauga,ChIJZ49FH3Q4K4gRMECgKM4_lYE,Maxwell Physiotherapy and Rehab Clinic,"3415 Fieldgate Dr, Mississauga, ON L4X 2J4, Ca...",43.624210,-79.588187,4.9,143.0,"establishment,health,physiotherapist,point_of_...",http://www.maxwellclinic.ca/,https://maps.google.com/?cid=9337439557099995184,POINT (-79.58819 43.62421),3289.0,2021S05075350526.02,5350526.02


In [ ]:
# Build a unique mapping of Place ID → DGUID (and CTUID if needed)
place_to_tract = (
    gdf_physio_DGUID[["Place ID", "DGUID", "CTUID"]]
    .dropna(subset=["DGUID"])
    .drop_duplicates(subset=["Place ID"])
    .reset_index(drop=True)
)

# Merge DGUID into pcr_df
pcr_with_dguid = pcr_df.merge(place_to_tract, on="Place ID", how="left")

print(" Reviews with DGUID shape:", pcr_with_dguid.shape)
pcr_with_dguid.head()


✅ Reviews with DGUID shape: (6839, 10)


,City,Place ID,Clinic Name,Author,Rating,Text,Time,Relative Time,DGUID,CTUID
0,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Deepak Kamani,5,I had a wonderful experience at Real rehab. Th...,1735358537,9 months ago,2021S05075350513.01,5350513.01
1,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Deep Inder,5,I can’t say enough good things about my massag...,1745077110,5 months ago,2021S05075350513.01,5350513.01
2,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Frankline Antony,5,“I had a fantastic experience.The physiotherap...,1751733894,2 months ago,2021S05075350513.01,5350513.01
3,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Katie C,5,I recently started seeing Reva for massage the...,1743888183,5 months ago,2021S05075350513.01,5350513.01
4,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,Monika Meharchandani,5,Visiting Real Rehab is always an exceptional e...,1742052614,6 months ago,2021S05075350513.01,5350513.01


In [ ]:
# Mapping support facilities to census tracts

# Convert support facilities DataFrame to GeoDataFrame (use sf_df, not pc_df!)
geometry = [Point(xy) for xy in zip(sf_df["Longitude"], sf_df["Latitude"])]
gdf_support = gpd.GeoDataFrame(sf_df.copy(), geometry=geometry, crs="EPSG:4326")

# Making sure the census tract data is in the same CRS
gdf_filtered_with_ids = gdf_filtered_with_ids.to_crs("EPSG:4326")

# Spatial join to get DGUID and other tract details
gdf_support_DGUID = gpd.sjoin(
    gdf_support,
    gdf_filtered_with_ids[["DGUID", "CTUID", "geometry"]],
    how="left",
    predicate="within"
)

# Drop unmatched rows
gdf_support_DGUID = gdf_support_DGUID.dropna(subset=["DGUID"])

# checking head
print(f"✅ Total matched support facilities: {gdf_support_DGUID.shape[0]}")
gdf_support_DGUID.head()


✅ Total matched support facilities: 939


,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL,geometry,index_right,DGUID,CTUID
0,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,"2200 Eglinton Ave W, Mississauga, ON L5M 2N1, ...",43.558609,-79.703268,3.0,1649.0,"establishment,health,hospital,point_of_interest",http://trilliumhealthpartners.ca/,https://maps.google.com/?cid=2655445724276753516,POINT (-79.70327 43.55861),113.0,2021S05075350516.31,5350516.31
1,Mississauga,ChIJDYSaI_ZGK4gRHtlzIG1irZU,Mississauga Hospital,"100 Queensway W, Mississauga, ON L5B 1B8, Canada",43.571699,-79.607584,2.6,1378.0,"establishment,health,hospital,point_of_interest",https://www.thp.ca/,https://maps.google.com/?cid=10785384903457626398,POINT (-79.60758 43.5717),3267.0,2021S05075350513.03,5350513.03
2,Mississauga,ChIJrSXDN6JnK4gR_Z2rizri6Ug,Oakville Trafalgar Memorial Hospital,"3001 Hospital Gate, Oakville, ON L6M 0L8, Canada",43.450353,-79.764738,2.9,1306.0,"establishment,health,hospital,point_of_interest",https://www.haltonhealthcare.on.ca/locations/o...,https://maps.google.com/?cid=5253979181383654909,POINT (-79.76474 43.45035),1394.0,2021S05075350615.00,5350615.00
2,Mississauga,ChIJrSXDN6JnK4gR_Z2rizri6Ug,Oakville Trafalgar Memorial Hospital,"3001 Hospital Gate, Oakville, ON L6M 0L8, Canada",43.450353,-79.764738,2.9,1306.0,"establishment,health,hospital,point_of_interest",https://www.haltonhealthcare.on.ca/locations/o...,https://maps.google.com/?cid=5253979181383654909,POINT (-79.76474 43.45035),1394.0,2021S05075350615.00,5350615.00
3,Mississauga,ChIJnTiQbHUxK4gRfP8pK7aXbxM,Humber River Hospital,"1235 Wilson Ave, North York, ON M3M 0B2, Canada",43.724222,-79.488578,2.6,2281.0,"establishment,health,hospital,point_of_interest",http://www.hrh.ca/,https://maps.google.com/?cid=1400504817799528316,POINT (-79.48858 43.72422),1598.0,2021S05075350290.01,5350290.01


In [ ]:
# Build mapping Place ID → DGUID (and CTUID if useful)
support_place_to_tract = (
    gdf_support_DGUID[["Place ID", "DGUID", "CTUID"]]
    .dropna(subset=["DGUID"])
    .drop_duplicates(subset=["Place ID"])
    .reset_index(drop=True)
)

# Merge DGUID into reviews table
sfr_with_DGUID = sfr_df.merge(support_place_to_tract, on="Place ID", how="left")

print("✅ Reviews with DGUID shape:", sfr_with_DGUID.shape)
sfr_with_DGUID.head()


✅ Reviews with DGUID shape: (6709, 10)


,City,Place ID,Facility Name,Author,Rating,Text,Time,Relative Time,DGUID,CTUID
0,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Tejal P,4,Nurses here are angels. Had a great experience...,1753800941,2 months ago,2021S05075350516.31,5350516.31
1,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Ram Venkat,4,My daughters delivery 27th July 2025\nGreat se...,1758075027,a week ago,2021S05075350516.31,5350516.31
2,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Ozair Mohammad,1,"Insane wait times, hasn’t gotten better in yea...",1749294804,3 months ago,2021S05075350516.31,5350516.31
3,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Sharry A,5,I had an emergency visit recently and was incr...,1746054361,5 months ago,2021S05075350516.31,5350516.31
4,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,Terrance 413,5,No one likes spending 6.5 hours in the Emergen...,1753756252,2 months ago,2021S05075350516.31,5350516.31


In [ ]:
# Checking number of Unique census tracts with DGUID
dguid_list = gdf_filtered_with_ids['DGUID'].unique().tolist()
print(f"Number of unique DGUIDs for selected cities: {len(dguid_list)}")

Number of unique DGUIDs for selected cities: 853


In [ ]:
# Checking unique Census Subdivisions for all Census Tracts
gdf_filtered_with_ids['City'].unique()

array(['Burlington', 'Mississauga', 'Toronto', 'Oakville'], dtype=object)

In [ ]:
from shapely.geometry import Polygon
import geopandas as gpd

# Defining Co-ordinates for Etobicoke Polygon
etobicoke_coords = [
    (43.6205, -79.5132), (43.6000, -79.5500), (43.6300, -79.5200),
    (43.6400, -79.5000), (43.6100, -79.4800), (43.5850, -79.5200),
    (43.5900, -79.5000), (43.6250, -79.5400)
]

etobicoke_poly = Polygon([(lon, lat) for lat, lon in etobicoke_coords])

In [ ]:
import geopandas as gpd

# --- 0) Inputs expected:
# gdf_filtered_with_ids  -> tracts for Mississauga/Oakville/Burlington/Toronto (EPSG:4326)
# etobicoke_poly         -> your shapely Polygon drawn in EPSG:4326 (lon, lat order inside)

# --- 1) Work in a projected CRS for robust geometry tests
CRS_PROJ = 32617  # UTM Zone 17N
gdf_proj = gdf_filtered_with_ids.to_crs(CRS_PROJ)

# Project the Etobicoke polygon too
etobicoke_poly_proj = gpd.GeoSeries([etobicoke_poly], crs="EPSG:4326").to_crs(CRS_PROJ).iloc[0]

# --- 2) Choose how to tag Etobicoke tracts
USE_INTERSECTS = True   # set False to use centroid-within (more strict)

if USE_INTERSECTS:
    # Any Toronto tract whose polygon touches the Etobicoke polygon
    mask_etobicoke = (gdf_proj["City"] == "Toronto") & (gdf_proj.geometry.intersects(etobicoke_poly_proj))
else:
    # Centroid must fall inside (optionally add a small buffer to be more inclusive)
    poly_for_centroids = etobicoke_poly_proj.buffer(100)  # meters; tweak or set to 0 if you want no buffer
    mask_etobicoke = (gdf_proj["City"] == "Toronto") & (gdf_proj.geometry.centroid.within(poly_for_centroids))

# --- 3) Relabel on the original frame using the SAME index (no concat/reset needed)
gdf_final = gdf_filtered_with_ids.copy()
gdf_final.loc[mask_etobicoke.index[mask_etobicoke], "City"] = "Etobicoke"

# Keep only target cities and return to WGS84
target_cities = ['Mississauga', 'Oakville', 'Burlington', 'Toronto', 'Etobicoke']
gdf_final = gdf_final[gdf_final["City"].isin(target_cities)].to_crs(epsg=4326)

# --- 4) Quick checks
print("Final unique census tracts:", gdf_final["CTUID"].nunique())
print(gdf_final["City"].value_counts())
gdf_final.head()


Final unique census tracts: 853
City
Toronto        583
Mississauga    167
Burlington      59
Oakville        57
Etobicoke       21
Name: count, dtype: int64


,CTUID,DGUID,CTNAME,LANDAREA,PRUID_left,geometry,index_right,CSDUID,City,CSDTYPE,PRUID_right,PRNAME,CDUID,CDNAME,CDTYPE
25,5370140.04,2021S05075370140.04,0140.04,3.0270,35,"POLYGON ((-79.89286 43.32909, -79.89226 43.328...",1847,3524002,Burlington,CY,35,Ontario,3524,Halton,RM
113,5350516.31,2021S05075350516.31,0516.31,1.4206,35,"POLYGON ((-79.69225 43.56952, -79.69207 43.569...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
118,5350516.32,2021S05075350516.32,0516.32,1.0444,35,"POLYGON ((-79.78141 43.57025, -79.78486 43.567...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
119,5350520.07,2021S05075350520.07,0520.07,0.3389,35,"POLYGON ((-79.61896 43.57806, -79.61944 43.577...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
120,5350520.08,2021S05075350520.08,0520.08,1.1020,35,"POLYGON ((-79.62281 43.57841, -79.62241 43.578...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM


In [ ]:
# Keep only Mississauga, Oakville, Burlington, Etobicoke
target_cities = ['Mississauga', 'Oakville', 'Burlington', 'Etobicoke']
gdf_final = gdf_final[gdf_final["City"].isin(target_cities)].copy()

print("Final unique census tracts:", gdf_final["CTUID"].nunique())
print(gdf_final["City"].value_counts())
gdf_final.head()


Final unique census tracts: 284
City
Mississauga    167
Burlington      59
Oakville        57
Etobicoke       21
Name: count, dtype: int64


,CTUID,DGUID,CTNAME,LANDAREA,PRUID_left,geometry,index_right,CSDUID,City,CSDTYPE,PRUID_right,PRNAME,CDUID,CDNAME,CDTYPE
25,5370140.04,2021S05075370140.04,0140.04,3.0270,35,"POLYGON ((-79.89286 43.32909, -79.89226 43.328...",1847,3524002,Burlington,CY,35,Ontario,3524,Halton,RM
113,5350516.31,2021S05075350516.31,0516.31,1.4206,35,"POLYGON ((-79.69225 43.56952, -79.69207 43.569...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
118,5350516.32,2021S05075350516.32,0516.32,1.0444,35,"POLYGON ((-79.78141 43.57025, -79.78486 43.567...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
119,5350520.07,2021S05075350520.07,0520.07,0.3389,35,"POLYGON ((-79.61896 43.57806, -79.61944 43.577...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM
120,5350520.08,2021S05075350520.08,0520.08,1.1020,35,"POLYGON ((-79.62281 43.57841, -79.62241 43.578...",1545,3521005,Mississauga,CY,35,Ontario,3521,Peel,RM


We will use **Statistics Canada 2021 census data** at the DGUID level. The processing steps include:

- Filtering census records to only the selected GTA census tracts
- Removing non-informative rate and percentage columns
- Reshaping the data into a wide format with one row per DGUID
- Selecting demographic and income variables relevant to physiotherapy demand

This will give us a clean, analysis-ready demographic dataset aligned perfectly with clinic locations

In [ ]:
# Defining path to Census Data
path_ct="/content/drive/MyDrive/Evodia_Tech/Arrow Physiotherapy/98-401-X2021007_English_CSV_data.csv"

In [ ]:
# Reading the dataset
df_ct=pd.read_csv(path_ct,encoding="ISO-8859-1")

/tmp/ipython-input-1662001239.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ct=pd.read_csv(path_ct,encoding="ISO-8859-1")


In [ ]:
# Checking first 5 rows of the dataset
df_ct.head()

,CENSUS_YEAR,DGUID,ALT_GEO_CODE,GEO_LEVEL,GEO_NAME,TNR_SF,TNR_LF,DATA_QUALITY_FLAG,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,...,C2_COUNT_MEN+,SYMBOL.1,C3_COUNT_WOMEN+,SYMBOL.2,C10_RATE_TOTAL,SYMBOL.3,C11_RATE_MEN+,SYMBOL.4,C12_RATE_WOMEN+,SYMBOL.5
0,2021,2021S0503932,932.0,Census metropolitan area,Abbotsford - Mission,2.7,3.7,0,1,"Population, 2021",...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
1,2021,2021S0503932,932.0,Census metropolitan area,Abbotsford - Mission,2.7,3.7,0,2,"Population, 2016",...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
2,2021,2021S0503932,932.0,Census metropolitan area,Abbotsford - Mission,2.7,3.7,0,3,"Population percentage change, 2016 to 2021",...,NaN,...,NaN,...,8.4,NaN,NaN,...,NaN,...
3,2021,2021S0503932,932.0,Census metropolitan area,Abbotsford - Mission,2.7,3.7,0,4,Total private dwellings,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
4,2021,2021S0503932,932.0,Census metropolitan area,Abbotsford - Mission,2.7,3.7,0,5,Private dwellings occupied by usual residents,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...


In [ ]:
# Checking shape of the dataset
df_ct.shape

(16567407, 23)

In [ ]:
#Checking info on the dataset
df_ct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16567407 entries, 0 to 16567406
Data columns (total 23 columns):
 #   Column               Dtype  
---  ------               -----  
 0   CENSUS_YEAR          int64  
 1   DGUID                object 
 2   ALT_GEO_CODE         float64
 3   GEO_LEVEL            object 
 4   GEO_NAME             object 
 5   TNR_SF               float64
 6   TNR_LF               float64
 7   DATA_QUALITY_FLAG    int64  
 8   CHARACTERISTIC_ID    int64  
 9   CHARACTERISTIC_NAME  object 
 10  CHARACTERISTIC_NOTE  float64
 11  C1_COUNT_TOTAL       float64
 12  SYMBOL               object 
 13  C2_COUNT_MEN+        float64
 14  SYMBOL.1             object 
 15  C3_COUNT_WOMEN+      float64
 16  SYMBOL.2             object 
 17  C10_RATE_TOTAL       float64
 18  SYMBOL.3             object 
 19  C11_RATE_MEN+        float64
 20  SYMBOL.4             object 
 21  C12_RATE_WOMEN+      float64
 22  SYMBOL.5             object 
dtypes: float64(10), int64(3), obje

In [ ]:
# Get DGUIDs from your selected tracts ---
dguid_list = gdf_final['DGUID'].unique().tolist()

# Filter census data for selected DGUIDs ---
df_filtered = df_ct[df_ct['DGUID'].isin(dguid_list)].copy()

# Dropping duplicate entries
df_filtered = df_filtered.drop_duplicates()

# Drop rate columns
columns_to_drop = ['C10_RATE_TOTAL', 'C11_RATE_MEN+', 'C12_RATE_WOMEN+']
df_filtered = df_filtered.drop(columns=columns_to_drop, errors='ignore')

# Step 1: Group to ensure uniqueness before pivoting
df_grouped = df_filtered.groupby(['DGUID', 'CHARACTERISTIC_NAME'])['C1_COUNT_TOTAL'].first().reset_index()

# Step 2: Pivot to wide format
df_pivot = df_grouped.pivot(
   index='DGUID',
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL'
).reset_index()

# Step 3: Merge with gdf_final (if needed)
merged_df = gdf_final.merge(df_pivot, on='DGUID', how='left')

print("Final shape:", merged_df.shape)
merged_df.head()



Final shape: (304, 1777)


,CTUID,DGUID,CTNAME,LANDAREA,PRUID_left,geometry,index_right,CSDUID,City,CSDTYPE,...,Total - Religion for the population in private households - 25% sample data,Total - Secondary (high) school diploma or equivalency certificate for the population aged 15 years and over in private households - 25% sample data,Total - Secondary (high) school diploma or equivalency certificate for the population aged 25 to 64 years in private households - 25% sample data,"Total - Tenant households in non-farm, non-reserve private dwellings - 25% sample data",Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Total - Total income groups in 2020 for the population aged 15 years and over in private households - 100% data,Total - Visible minority for the population in private households - 25% sample data,Total number of census families in private households - 100% data,Total private dwellings,Unemployment rate
0,5370140.04,2021S05075370140.04,0140.04,3.0270,35,"POLYGON ((-79.89286 43.32909, -79.89226 43.328...",1847,3524002,Burlington,CY,...,3905.0,3100.0,2040.0,100.0,1175.0,3120.0,3905.0,1160.0,1382.0,10.6
1,5350516.31,2021S05075350516.31,0516.31,1.4206,35,"POLYGON ((-79.69225 43.56952, -79.69207 43.569...",1545,3521005,Mississauga,CY,...,2770.0,2365.0,1415.0,85.0,695.0,2370.0,2770.0,820.0,941.0,11.2
2,5350516.32,2021S05075350516.32,0516.32,1.0444,35,"POLYGON ((-79.78141 43.57025, -79.78486 43.567...",1545,3521005,Mississauga,CY,...,5095.0,4225.0,2910.0,55.0,1580.0,4145.0,5095.0,1415.0,1447.0,13.0
3,5350520.07,2021S05075350520.07,0520.07,0.3389,35,"POLYGON ((-79.61896 43.57806, -79.61944 43.577...",1545,3521005,Mississauga,CY,...,4955.0,3890.0,2835.0,1140.0,1280.0,3870.0,4955.0,1300.0,2014.0,18.1
4,5350520.08,2021S05075350520.08,0520.08,1.1020,35,"POLYGON ((-79.62281 43.57841, -79.62241 43.578...",1545,3521005,Mississauga,CY,...,6090.0,5270.0,3615.0,760.0,2030.0,5265.0,6095.0,1720.0,2354.0,14.3


In [ ]:
df_pivot.columns = [col.strip().replace('\n', ' ') for col in df_pivot.columns]


In [ ]:
for i, col in enumerate(merged_df.columns, start=1):
    print(f"{i}. {col}")

1. CTUID
2. DGUID
3. CTNAME
4. LANDAREA
5. PRUID_left
6. geometry
7. index_right
8. CSDUID
9. City
10. CSDTYPE
11. PRUID_right
12. PRNAME
13. CDUID
14. CDNAME
15. CDTYPE
16.                 Bosnian
17.                 Croatian
18.                 Dari
19.                 Iranian Persian
20.                 Odia
21.                 Oriya, n.o.s.
22.                 Persian (Farsi), n.o.s.
23.                 Serbian
24.                 Serbo-Croatian, n.i.e.
25.               Afrikaans
26.               Anishinaabemowin (Chippewa)
27.               Aramaic, n.o.s.
28.               Assamese
29.               Assyrian Neo-Aramaic
30.               Baluchi
31.               Belarusian
32.               Bengali
33.               Bosnian
34.               Bulgarian
35.               Chaldean Neo-Aramaic
36.               Cree, n.o.s.
37.               Croatian
38.               Czech
39.               Daawaamwin (Odawa)
40.               Danish
41.               Dari
42.               Deh G

In [ ]:
# List of column indices you want to keep
columns_to_keep = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                   1682, 1135, 1143, 1174,1179, 1031,
                   1032, 923, 924, 1671, 1672,1674, 1776,
                   926, 1354, 1555, 1664]

# Select columns by index
df_reduced = merged_df.iloc[:, columns_to_keep]

Keeping client requirement in perspective, We will extarct only a handful of features to use for analysis and feature engineering.

In [ ]:
column_names = merged_df.columns[columns_to_keep]
print(column_names)


Index(['CTUID', 'DGUID', 'CTNAME', 'LANDAREA', 'PRUID_left', 'geometry',
       'index_right', 'CSDUID', 'City', 'CSDTYPE',
       'Total - Age groups of the population - 100% data', '  0 to 14 years',
       '  15 to 64 years', '  65 years and over', '  85 years and over',
       '    Median total income in 2019 among recipients ($)',
       '    Median total income in 2020 among recipients ($)',
       '    Average total income in 2019 among recipients ($)',
       '    Average total income in 2020 among recipients ($)',
       'Population density per square kilometre',
       'Population percentage change, 2016 to 2021', 'Population, 2021',
       'Unemployment rate', '    Bachelor's degree or higher',
       '  High (secondary) school diploma or equivalency certificate',
       '  Postsecondary certificate, diploma or degree', 'Employment rate'],
      dtype='object')


In [ ]:
df_reduced.head()

,CTUID,DGUID,CTNAME,LANDAREA,PRUID_left,geometry,index_right,CSDUID,City,CSDTYPE,...,Average total income in 2019 among recipients ($),Average total income in 2020 among recipients ($),Population density per square kilometre,"Population percentage change, 2016 to 2021","Population, 2021",Unemployment rate,Bachelor's degree or higher,High (secondary) school diploma or equivalency certificate,"Postsecondary certificate, diploma or degree",Employment rate
0,5370140.04,2021S05075370140.04,0140.04,3.0270,35,"POLYGON ((-79.89286 43.32909, -79.89226 43.328...",1847,3524002,Burlington,CY,...,68500.0,71300.0,1276.5,6.3,3864.0,10.6,1230.0,735.0,2075.0,65.5
1,5350516.31,2021S05075350516.31,0516.31,1.4206,35,"POLYGON ((-79.69225 43.56952, -79.69207 43.569...",1545,3521005,Mississauga,CY,...,61250.0,61600.0,1909.8,-3.2,2713.0,11.2,1140.0,485.0,1675.0,51.9
2,5350516.32,2021S05075350516.32,0516.32,1.0444,35,"POLYGON ((-79.78141 43.57025, -79.78486 43.567...",1545,3521005,Mississauga,CY,...,56550.0,59650.0,4759.7,0.4,4971.0,13.0,1960.0,920.0,2820.0,62.1
3,5350520.07,2021S05075350520.07,0520.07,0.3389,35,"POLYGON ((-79.61896 43.57806, -79.61944 43.577...",1545,3521005,Mississauga,CY,...,37280.0,42880.0,14544.1,-5.2,4929.0,18.1,1905.0,765.0,2590.0,53.9
4,5350520.08,2021S05075350520.08,0520.08,1.1020,35,"POLYGON ((-79.62281 43.57841, -79.62241 43.578...",1545,3521005,Mississauga,CY,...,44080.0,48320.0,5529.9,-3.3,6094.0,14.3,1930.0,1370.0,3155.0,56.0


In [ ]:
df_reduced.describe().T

,count,mean,std,min,25%,50%,75%,max
LANDAREA,304.0,5.105493,16.321362,0.0798,0.9981,1.4524,2.823225,126.5048
index_right,304.0,1659.983553,146.758855,1544.0000,1545.0000,1545.0000,1846.000000,1847.0000
Total - Age groups of the population - 100% data,302.0,4703.642384,2045.906531,45.0000,3705.0000,4660.0000,5658.750000,21315.0000
0 to 14 years,302.0,748.443709,481.620219,0.0000,531.2500,697.5000,900.000000,5470.0000
15 to 64 years,302.0,3156.572848,1428.858915,15.0000,2391.2500,3065.0000,3757.500000,14455.0000
65 years and over,302.0,798.692053,388.568625,5.0000,550.0000,765.0000,972.500000,2660.0000
85 years and over,300.0,54.433333,52.642661,0.0000,20.0000,40.0000,70.000000,325.0000
Median total income in 2019 among recipients ($),298.0,42070.469799,8848.736358,22800.0000,35200.0000,42000.0000,48800.000000,75500.0000
Median total income in 2020 among recipients ($),298.0,43863.087248,7683.892366,27600.0000,37600.0000,43800.0000,49600.000000,74000.0000
Average total income in 2019 among recipients ($),298.0,60355.000000,23926.715712,28440.0000,45970.0000,56575.0000,67050.000000,209800.0000


We can now save intermediate datasets as reusable artifacts for downstream analysis and dashboarding:

- Census demographic data mapped to DGUIDs
- Physiotherapy clinics mapped to census tracts
- Hospitals and walk-in clinics mapped to census tracts

These artifacts will be loaded directly by the Streamlit app, ensuring fast and responsive dashboard performance.

In [ ]:
# Save df_reduced as CSV
df_reduced.to_csv(f"{save_path}/df_reduced.csv", index=False)

# Save gdf_physio_DGUID as GeoJSON
gdf_physio_DGUID.to_file(f"{save_path}/gdf_physio_DGUID.geojson", driver='GeoJSON')

# Save gdf_hospitals_DGUID as GeoJSON
gdf_support_DGUID.to_file(f"{save_path}/gdf_hospitals_DGUID.geojson", driver='GeoJSON')

In [ ]:
# Save pcr_with_DGUID as CSV
pcr_with_dguid.to_csv(f"{save_path}/pcr_with_DGUID.csv", index=False)

# Save sfr_with_DGUID as CSV
sfr_with_DGUID.to_csv(f"{save_path}/sfr_with_DGUID.csv", index=False)

In [ ]:
import geopandas as gpd

# Load the saved GeoJSONs
gdf_physio_check = gpd.read_file("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/gdf_physio_DGUID.geojson")
gdf_support_check = gpd.read_file("/content/drive/MyDrive/Evodia_Tech/Arrow_Physiotherapy/gdf_hospitals_DGUID.geojson")

# Check basic info
print("Physio clinics:", gdf_physio_check.shape)
print("Support facilities:", gdf_support_check.shape)

# Optional: show first few rows
display(gdf_physio_check.head())
display(gdf_support_check.head())

# Check geometry types
print("Physio geometry types:", gdf_physio_check.geometry.type.unique())
print("Support geometry types:", gdf_support_check.geometry.type.unique())


Physio clinics: (968, 15)
Support facilities: (939, 15)


,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL,index_right,DGUID,CTUID,geometry
0,Mississauga,ChIJdY4UQJJHK4gRK6q4aZpXC0E,Real Rehab - The Hybrid Physiotherapy Clinic i...,"1224 Dundas St W, Mississauga, ON L5C 4G7, Canada",43.553812,-79.645781,4.8,73.0,"establishment,health,point_of_interest",https://real-rehab.com/,https://maps.google.com/?cid=4686936157919423019,3265.0,2021S05075350513.01,5350513.01,POINT (-79.64578 43.55381)
1,Mississauga,ChIJC7CuKt81K4gRorDvzCPO2bY,Vibrant Physiotherapy,"3050 Confederation Pkwy #206, Mississauga, ON ...",43.577976,-79.621476,4.9,36.0,"establishment,health,point_of_interest",http://www.vibrantphysiotherapy.ca/,https://maps.google.com/?cid=13175788838006534306,120.0,2021S05075350520.08,5350520.08,POINT (-79.62148 43.57798)
2,Mississauga,ChIJgb1mccFHK4gR_6BEh0rlLJ0,Proremedy Physiotherapy,"190 Robert Speck Pkwy Suite 200, Mississauga, ...",43.596916,-79.632251,4.9,171.0,"establishment,health,physiotherapist,point_of_...",https://proremedyphysio.com/physiotherapy-miss...,https://maps.google.com/?cid=11325679271189717247,4827.0,2021S05075350527.13,5350527.13,POINT (-79.63225 43.59692)
3,Mississauga,ChIJkZx3yb1DK4gRU2szteXkKgc,Smart Physiotherapy Clinic,"1151 Dundas St W, Mississauga, ON L5C 1C4, Canada",43.557562,-79.645423,2.5,31.0,"establishment,health,point_of_interest",http://www.paradisewelness.ca/,https://maps.google.com/?cid=516476782526032723,1240.0,2021S05075350518.00,5350518.00,POINT (-79.64542 43.55756)
4,Mississauga,ChIJZ49FH3Q4K4gRMECgKM4_lYE,Maxwell Physiotherapy and Rehab Clinic,"3415 Fieldgate Dr, Mississauga, ON L4X 2J4, Ca...",43.624210,-79.588187,4.9,143.0,"establishment,health,physiotherapist,point_of_...",http://www.maxwellclinic.ca/,https://maps.google.com/?cid=9337439557099995184,3289.0,2021S05075350526.02,5350526.02,POINT (-79.58819 43.62421)


,City,Place ID,Name,Address,Latitude,Longitude,Rating,User Ratings Total,Types,Website,Google Maps URL,index_right,DGUID,CTUID,geometry
0,Mississauga,ChIJd4VZ3IVBK4gRbITYK9EJ2iQ,Credit Valley Hospital,"2200 Eglinton Ave W, Mississauga, ON L5M 2N1, ...",43.558609,-79.703268,3.0,1649.0,"establishment,health,hospital,point_of_interest",http://trilliumhealthpartners.ca/,https://maps.google.com/?cid=2655445724276753516,113.0,2021S05075350516.31,5350516.31,POINT (-79.70327 43.55861)
1,Mississauga,ChIJDYSaI_ZGK4gRHtlzIG1irZU,Mississauga Hospital,"100 Queensway W, Mississauga, ON L5B 1B8, Canada",43.571699,-79.607584,2.6,1378.0,"establishment,health,hospital,point_of_interest",https://www.thp.ca/,https://maps.google.com/?cid=10785384903457626398,3267.0,2021S05075350513.03,5350513.03,POINT (-79.60758 43.5717)
2,Mississauga,ChIJrSXDN6JnK4gR_Z2rizri6Ug,Oakville Trafalgar Memorial Hospital,"3001 Hospital Gate, Oakville, ON L6M 0L8, Canada",43.450353,-79.764738,2.9,1306.0,"establishment,health,hospital,point_of_interest",https://www.haltonhealthcare.on.ca/locations/o...,https://maps.google.com/?cid=5253979181383654909,1394.0,2021S05075350615.00,5350615.00,POINT (-79.76474 43.45035)
3,Mississauga,ChIJrSXDN6JnK4gR_Z2rizri6Ug,Oakville Trafalgar Memorial Hospital,"3001 Hospital Gate, Oakville, ON L6M 0L8, Canada",43.450353,-79.764738,2.9,1306.0,"establishment,health,hospital,point_of_interest",https://www.haltonhealthcare.on.ca/locations/o...,https://maps.google.com/?cid=5253979181383654909,1394.0,2021S05075350615.00,5350615.00,POINT (-79.76474 43.45035)
4,Mississauga,ChIJnTiQbHUxK4gRfP8pK7aXbxM,Humber River Hospital,"1235 Wilson Ave, North York, ON M3M 0B2, Canada",43.724222,-79.488578,2.6,2281.0,"establishment,health,hospital,point_of_interest",http://www.hrh.ca/,https://maps.google.com/?cid=1400504817799528316,1598.0,2021S05075350290.01,5350290.01,POINT (-79.48858 43.72422)


Physio geometry types: ['Point']
Support geometry types: ['Point']
